# JobPulse  Documentation
=======================================================================================================================================



## 1. Business Understanding

### 1.1 Business Objective
African tech job seekers lack visibility into what skills employers are actually demanding, how fast that demand is shifting, and how their own CV stacks up against the market. JobPulse addresses this by continuously collecting job postings from African job boards and turning them into actionable career intelligence.

### 1.2 Objectives
- Give job seekers a real-time view of the most in-demand and fastest-growing technical skills in the African tech market.
- Let a user upload their CV and get a market-match score, a list of skills they already have, and a ranked list of missing skills to prioritize learning.
- Surface salary benchmarks and remote-work prevalence per skill/role.
- Provide a RAG-based AI assistant that answers market questions grounded in real job data (e.g. *"Which remote Python jobs are available in Kenya?"*).
- Career path recommendations.

### 1.3 Success Criteria
The CV-to-job match score should achieve **at least 90% accuracy** - i.e., the computed match percentage should reliably reflect how well a candidate's actual skills/experience align with a job's requirements, validated against a reference set of manually-assessed CV/job pairs.


### 1.4 Project Plan / Scope
- **In scope (built):** automated scraping pipeline (14 sources), staged ingestion/cleaning/NLP/analytics pipeline, dashboard analytics, CV-to-market matching engine, React frontend, RAG retrieval for the AI assistant,Docker deployment of the pipeline.
- **Out of scope / future:** trained ML skill-category classifier, unified FastAPI backend for the pipeline itself, a generation/LLM layer on top of RAG retrieval,  Career Insights.



---

## 2. Data Understanding

### 2.1 Data Collection
Two repos are involved in getting data into the product:

- **jobpulseKe/ pipeline repo** - the actual collection system. Site-specific scrapers live under **src/collectors/** (Requests + BeautifulSoup + Tenacity for retry/politeness), orchestrated by **scripts/run_scrapers.py**. Each scraper writes its own **<source> csv**; these are merged and deduplicated on **job_id** via **scripts/merge_csvs.py**, with **scripts/merge_jobpulseke.py** folding in a sister Kenya-specific dataset and **scripts/merge_public_datasets.py** optionally supplementing with public Hugging Face job datasets.
- Merged output feeds a master file, **data/external/jobpulseke_master_africa_tech_jobs.csv** (22 columns, schema defined in **src/config.py** / **src/scraping_config.py**), which is what Stage 1 ingestion reads.
- Collection can be run ad hoc (**python scripts/run_scrapers.py**) or on a lock-protected cron schedule (**scripts/run_scheduled_scrape.py**, template at **cron/jobpulse-scrapers.cron.example**) that automatically chains Stage 1–4 after every successful scrape.
- The **app repo** (see section 6.2) also references a **cron/** folder for automated scraping and reads from a **jobpulse_cleaned.csv** - this is likely a CSV export of the pipeline's cleaned Parquet output.


### 2.2 Data Description
Dataset snapshot previously analyzed for the dashboard: **jobpulse_cleaned.csv**, market data collected **2026-08-31**, **10,379 postings**. 

| Column | Description |
|---|---|
| job_id | Unique identifier for the posting |
| source | Job board the posting was scraped from |
| source_job_id | Original ID/slug on the source site |
| job_title | Title of the role |
| company | Employer name |
| job_description| Full posting text |
| location, country | Geographic fields |
| work_mode, remote_scope| Onsite/remote/hybrid classification |
| job_field, industry | Category classification |
| employment_type | Full-time / contract / etc. |
| experience_required, education_required | Candidate requirements |
| salary, currency | Compensation, where disclosed |
| date_posted, application_deadline, scraped_at | Timestamps |
| tech_category | Derived/NLP-tagged technology category |
| vacancy_url | Link to original posting |

### 2.3 Data Quality Notes
Computed from the **jobpulse_cleaned.csv** snapshot (10,379 rows):

| Field | % Missing |
|---|---|
| salary | 98.3% |
| currency | 99.4% |
| tech_category | 23.4% |
| company | 24.3% |
| location | 25.1% |
| job_field | 91.0% |
| industry | 91.5% |
| employment_type | 92.2% |
| experience_required | 94.8% |
| education_required | 93.7% |

**Key observations:**
- **Salary sparsity is severe** (98.3% missing) - consistent with the norm in the African job market where pay is rarely disclosed publicly. This limits how much confidence can be placed in the dashboard's "Average tech salary" figure.
- **tech_category is the field actually driving classification**, not **job_field** (91% empty, inconsistent free-text where populated). **tech_category** is populated for 76.6% of rows.
- **Postings are aggregated from 14 sources**, dominated by **fantastic_jobs_hf** (52%) and **jobberman** (19%), with smaller contributions from Kenya-specific boards (**myjobmag**, **brightermonday**, **fuzu**, **jobwebkenya**) and international/remote boards (**linkedin**, **indeed**, **remoteok**, **weworkremotely**). 



---

## 3. Data Preparation

Data preparation is staged and script-driven in the pipeline repo:

- **Stage 0 - Data Collection**: scrapers under src/collectors/, run via scripts/run_scrapers.py.

- **Stage 1 - Ingestion**: scripts/run_stage1_ingestion.py loads the merged master CSV, validates it against the schema in src/config.
py, fills empty descriptions, filters to African/remote-eligible postings, and saves timestamped output to data/processed/ as Parquet.

- **Stage 2 - Cleaning, Geo-Normalization & Deduplication**: scripts/run_stage2_cleaning.py standardizes country/city names, converts dates to ISO-8601, and fuzzy-match deduplicates, exporting data/processed/jobpulse_cleaned_*.parquet.

- **Stage 3 - NLP Pipeline & Skill Extraction**: src.nlp.run_stage_3_nlp_extraction - batched skill extraction (languages, frameworks, cloud, databases, AI tools) plus structured metadata extraction (years of experience, education, certifications, seniority). No CLI wrapper exists yet; it's called from a notebook or a custom script against an input/output Parquet path.

- **Stage 4 - Feature Engineering & Analytics**: src.analytics.stage4_orchestrator.run_stage_4_analytics - engineers features like salary_min/max_usd, is_remote, experience_bucket, seniority_order, and generates the Skill×Region matrix, salary distributions, and career pathways, exported as JSON/CSV under data/analytics/. Also no CLI wrapper.



---

## 4. Modeling


### 4.1 Pipeline repo - CV-driven job recommender (jobpulseKe/)
- scripts/recommend_from_cv.py parses an uploaded (.txt/.pdf/.docx) CV into a normalized skill profile, ranks current JobPulse records by required-skill coverage, and returns skill gaps, learning resources, and mock-interview prep. Jobs with a deadline in the next 7 days are flagged for priority action.

- Also exposed as an API: uvicorn src.api.recommender_api:app --reload → POST /recommend (accepts a CV upload, jobs_path, optional top_k). Uploaded CVs are written to a temp file for text extraction only and deleted before the response returns.

- Also reachable via the unified CLI: python scripts/jobpulse.py recommend --cv ... --jobs ... --top-k 10.

### 4.2 App repo - CV Score (0 - 100)
- A deterministic composite score computed from extracted signals (skill breadth, experience, education, certifications), explicitly labeled (score_type: "deterministic_composite") in API responses.


### 4.3 Skill/Metadata Extraction
- **App repo**: rule-based regex/keyword matching (app/ml/preprocessing/skill_extractor.py).
- **Pipeline repo **: broader batched extraction across languages, frameworks, cloud, databases, and AI tools, plus structured fields (experience, education, certifications, seniority) -src.nlp.run_stage_3_nlp_extraction.

### 4.4 Tech-Category Classification 
- Architecture is a TF-IDF + Logistic Regression classifier (app repo), designed to tag postings/CVs with a tech category. No trained model artifact exists yet (classifier_available: false at runtime). This aligns with the pipeline repo's own roadmap, which lists **Stage 5 — ML Classification** as planned/not yet implemented  the two repos agree this piece isn't done.

### 4.5 Job Recommendation Engine (app repo) 
- A hybrid weighted scorer (JobRecommender) matching candidates to jobs based on skills, experience, and location.

### 4.6 Time-Series / Job-Availability Forecasting 
- No forecasting model was built (app repo). Descoped after determining  date_posted coverage wasn't reliable enough. Replaced with CRUD-based job status tracking (JobStatusHistory) and a market-composition report.

### 4.7 AI Assistant / RAG retrieval 

- **Indexing**: scripts/build_rag_index.py - builds a vector index from the latest Stage 3 NLP output via a JobPulseRAG class (src/rag/retriever.py). Embedder backend defaults to auto - tries sentence-transformers first, falls back to offline TF-IDF if unavailable - or can be forced with --embedder tfidf / --embedder sentence-transformer.

- **Querying**: scripts/query_rag.py "<free-text query>" --top-k N returns ranked job postings (title, company, location, skills, similarity score, vacancy URL) for a natural-language query, e.g. "remote python developer in Kenya". Includes input validation (QueryValidationError) and a low-confidence check (has_strong_matches()) that warns the user when no result is a strong match rather than silently returning weak ones.
- This is the real implementation behind the Quick Start's python scripts/jobpulse.py ask "..." command referenced in section 6.1 that command is functional, not aspirational.

- **App repo**: still out of scope for the current backend; chromadb, qdrant-client, sentence-transformers are in requirements.txt but not wired up. Adapter pattern left in app/ml/registry/ for future extension the pipeline repo's implementation above is not yet connected to the app.

---

## 5. Evaluation

Evaluation is scripted, via two entry points in the pipeline repo:

### 5.1 scripts/evaluate_all_models.py - full model suite evaluation
Evaluates six components against a gold standard set (**src.evaluation.build_gold_set**, configurable sample size via --samples, default 30):

1. **Skill Extractor** - precision, recall, F1 against the gold set.
2. **Metadata Extractor** - accuracy on seniority, work-mode, and employment-type classification.
3. **Job Recommender** - top-1 accuracy and mean rank of the best candidate (**evaluate_job_recommender()**).
4. **RAG Retrieval** - avg MRR, avg Precision@5, and a low-confidence rate (**evaluate_rag_retrieval()**, skippable with --no-rag).
5. **Skill Normalizer** - alias-resolution accuracy.
6. **Skill Matcher** - separate match accuracy and score accuracy.

Output: a CSV and Markdown report per run (src.evaluation.report.generate_csv / generate_markdown), plus optional raw JSON with **--save-json**.


### 5.2 scripts/evaluate_nlp.py - NLP pipeline evaluation
A separate, narrower harness (src.nlp.evaluation.run_evaluation) specific to Stage 3, using three distinct checks rather than one blended score:
- **Coverage** over the real enriched dataset (how much of the corpus the extractor actually produces output for).
- **Title/seniority rule-based consistency** (an internal consistency check, not a gold-set comparison).
- **Precision/recall/F1** against a synthetic gold set (separate from the gold set used in 5.1).

Results can optionally be saved as JSON under data/nlp/evaluation/ with --save.

### 5.3 What this answers from the open evaluation questions
-  **Skill classifier/extractor validation** - has a concrete method now: precision/recall/F1 against a held-out gold set (5.1 #1), plus a separate coverage + consistency check specific to Stage 3 (5.2).
-  **Ranking/recommender validation** - top-1 accuracy and mean rank of best, against real candidates (5.1 #3).
-  **CV match score validation (Section 1.3 success criterion)** - only partially answered. The job recommender eval is the closest proxy, but the 90% success criterion in  was framed around the app repo's CV-to-job match **score** 

---


## 6. Deployment

### 6.1 Pipeline repo (jobpulseKe/) - data engineering layer
```
Scrapers (src/collectors/) 
   → run_scrapers.py → merged master CSV
   → Stage 1 ingestion → Parquet (data/processed/)
   → Stage 2 cleaning/geo-norm/dedup → Parquet
   → Stage 3 NLP skill extraction
   → Stage 4 feature engineering & analytics → JSON/CSV (data/analytics/)
```
- **Orchestration**: python scripts/jobpulse.py refresh --max-pages 5 runs the full pipeline (ingestion → cleaning → NLP → feature engineering → analytics) against the currently activated Python env (venv or Conda).

- **CV recommendation**: python scripts/jobpulse.py recommend --cv ... --jobs ... --top-k 10.

- **RAG query** : python scripts/jobpulse.py ask "...".

- **Scheduling**: scripts/run_scheduled_scrape.py, lock-protected, template cron job every 6 hours (Nairobi time) at cron/jobpulse-scrapers.cron.example - set JOBPULSE_ROOT and JOBPULSE_PYTHON before installing with crontab.

- **Stages 5–9 (ML classification, BI/viz exports, FastAPI + Streamlit app, Dockerized deployment)**: scaffolding exists under src/models/, src/api/, src/rag/ 

- **Tech stack**: Requests/BeautifulSoup/Tenacity (scraping), Pandas/Polars/PyArrow (processing), SpaCy/Sentence-Transformers/Hugging Face (NLP), Scikit-learn/LightGBM (ML, planned), PostgreSQL/PGVector (planned), ChromaDB/Qdrant (planned), FastAPI (planned), Streamlit (planned), Docker (planned).

### 6.2 App repo - product layer
```
React Frontend (Vite, Tailwind CSS)
      |
   FastAPI  ── PostgreSQL (users, resumes, jobs, skills, recommendations, model_predictions)
      |     ── Redis + Celery (async CV analysis, job ingestion, stale-job sweep)
      |
      ├── skill/metadata extractor (rule-based)
      ├── recommendation engine (deterministic hybrid scorer)
      └── job status service (CRUD, no forecasting)
```
- **Backend**: FastAPI app (ui/jobpulse-backend/jobpulse-backend/), with a model registry (app/ml/registry/) reporting real component status (READY / DEGRADED / NOT_AVAILABLE) via GET /health and GET /api/models.

- **Frontend**: React + Vite (ui/jobpulse-unified-app2.0/), Tailwind CSS. Built: Dashboard, CV Analyzer, Skill Demand. Planned: Career Insights, Salary Insights, AI Assistant.

- **Running it**: Docker (docker compose up --build - API, Celery worker/beat, PostgreSQL, Redis) or local (pip install -r requirements.txt, set DATABASE_URL/SECRET_KEY, uvicorn app.main:app --reload). Tests: pytest tests/ -v against disposable SQLite.

- **Current status**: frontend runs locally (npm run dev, localhost:5173). Backend runnable locally/via Docker.

### 6.3 Reports
Earlier versions of the pipeline's Stages 1–4 wrote timestamped JSON report files into reports/ on every run (Stage 1 also produced ad-hoc Markdown/.txt verification write-ups). That report-writing logic has been removed - each stage now only prints its summary to the console, and Stage 4 still exports its real analytics tables as JSON/CSV. Old files in reports/ are kept for reference but won't grow further.

### 6.4 Monitoring & Maintenance
- Pipeline: automated scraping via cron keeps the dataset current; each stage's console summary (records in/out, columns, reduction rate) is the main run-time signal since dedicated report files were removed.
- App: component health observable at runtime via GET /health and`GET /api/models.


---

## Appendix: Screenshots Referenced
- Dashboard (market stats, most demanded/fastest growing skill, avg salary, remote %)
- CV Analyzer - Overview tab (match %, skills found, missing skills)
- CV Analyzer - Opportunity callout ("Learning SQL could unlock +1 job...")
- Skill Demand table (skill, demand %, growth %, job count, avg salary, popular role, trend)

===========================================================================================================================================